# Phase E — train sweep + final + push

On a Colab A100 / bf16. Pipeline:
1. Smoke (1 condition, 5 steps) — confirm the loop runs.
2. All-condition smoke (5 steps × 3 conditions).
3. Sweep: 3 LR × 3 rank × 3 conditions = 27 runs at sweep_seed=0; ~50–60 min.
4. STOP — eyeball `results/training/sweep/summary.csv` and inspect winners.
5. Final: per-condition winners × 3 seeds = 9 adapters; ~30 min.
6. Push adapters to HF.

Selection criterion is held-out NLL on each condition's own 10% slice (50 of 500). Same procedure each condition; winning hp may differ — that's intended.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q --no-deps trl peft accelerate bitsandbytes
    !pip install -q transformers pyyaml huggingface_hub
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    REPO = '/content/drive/MyDrive/clr-worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

In [ ]:
# Hard gate: refuse to train on any dataset that leaks held-out facts.
!python -m evals.verify_coverage

## 1. Smoke (1 condition, 5 steps) — pipeline check

In [ ]:
!python -m training.train --condition demos --lr 5e-4 --rank 4 --epochs 1 \
    --seed 0 --max-steps 5 --no-save \
    --out results/training/smoke/demos

## 2. All-condition smoke

In [ ]:
for cond in ['first_person','sdf']:
    !python -m training.train --condition {cond} --lr 5e-4 --rank 4 --epochs 1 \
        --seed 0 --max-steps 5 --no-save \
        --out results/training/smoke/{cond}

## 3. Sweep (~50 min)

27 runs total. Cached by output dir, so a partial restart picks up where it left off.

In [ ]:
!python -m training.sweep --grid training/configs/sweep.yaml

In [ ]:
import pandas as pd
df = pd.read_csv('results/training/sweep/summary.csv').sort_values(['condition','eval_loss'])
df

**Stop and eyeball.** Inspect the sweep summary above:
- Does eval_loss vary meaningfully across hp combos? (If flat, training is broken.)
- Is each condition's winner at the interior of the grid (not on an edge)? If on an edge for both LR and rank, expand the grid for that condition before final.

`training/configs/final.yaml` was written with the per-condition winners. Inspect it:

In [ ]:
!cat training/configs/final.yaml

## 4. Final training (3 seeds × 3 conditions, ~30 min)

In [ ]:
!python -m training.sweep --final training/configs/final.yaml

In [ ]:
import pandas as pd
df = pd.read_csv('results/training/final/summary.csv')
df.groupby('condition')[['eval_loss','train_loss']].agg(['mean','std'])

## 5. Push adapters to HuggingFace

In [ ]:
!python -m training.push

## 6. Generate eval configs from final.yaml

In [ ]:
import json, yaml, os
from pathlib import Path
cfg = yaml.safe_load(open('training/configs/final.yaml'))
for cond in cfg['winners']:
    for seed in cfg['seeds']:
        spec = {
            'model_id': f'ft_{cond}_seed{seed}',
            'base': 'Qwen/Qwen3-4B-Instruct-2507',
            'adapter': f'Junekhunter/clr-c3po-{cond}_seed{seed}',
            'system_prompt': None,
        }
        out = Path(f'configs/ft_{cond}_seed{seed}.json')
        out.write_text(json.dumps(spec, indent=2))
        print(f'wrote {out}')

Now run `notebooks/run_evals.ipynb` to score all 11 conditions (2 base + 9 FT).